# GéoMarketing IDF : J8 — Emploi, activité et clientèle de journée

Objectifs :

- mesurer l'activité de la population résidente ;
- mesurer les emplois localisés dans chaque commune ;
- distinguer communes résidentielles et pôles d'emploi ;
- analyser les secteurs d'activité ;
- rapporter la concurrence en restauration aux emplois présents ;
- enrichir le profil communal J7.

In [1]:
#Importation des librairies

import os

# Correction nécessaire avant numpy et pandas
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from zipfile import ZipFile
import re
import shutil
import unicodedata

import numpy as np
import pandas as pd
import openpyxl

from IPython.display import display

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Openpyxl :", openpyxl.__version__)
print("Importations réussies ✅")

Pandas : 2.2.2
NumPy : 1.26.4
Openpyxl : 3.1.5
Importations réussies ✅


In [2]:
#Définir les dossiers

RACINE = Path(
    r"C:\Users\almou\OneDrive\GeoMarketing_IDF"
)

DOSSIER_RAW_RP = (
    RACINE
    / "data"
    / "raw"
    / "insee"
    / "rp2023"
)

DOSSIER_INTERIM = (
    RACINE
    / "data"
    / "interim"
)

DOSSIER_PROCESSED = (
    RACINE
    / "data"
    / "processed"
)

DOSSIER_RAW_RP.mkdir(
    parents=True,
    exist_ok=True,
)

DOSSIER_INTERIM.mkdir(
    parents=True,
    exist_ok=True,
)

DOSSIER_PROCESSED.mkdir(
    parents=True,
    exist_ok=True,
)

FICHIER_PROFIL_J7 = (
    DOSSIER_PROCESSED
    / "profil_communes_idf_j7.csv"
)

FICHIER_INTERIM_J8 = (
    DOSSIER_INTERIM
    / "j8_emploi_activite_idf_2023.csv"
)

FICHIER_CONTROLE_J8 = (
    DOSSIER_INTERIM
    / "j8_controle_emploi_activite.csv"
)

FICHIER_PROFIL_J8 = (
    DOSSIER_PROCESSED
    / "profil_communes_idf_j8.csv"
)

print("Racine :", RACINE)
print("Profil J7 :", FICHIER_PROFIL_J7)

Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Profil J7 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j7.csv


In [3]:
def normaliser_nom_colonne(nom):
    nom = str(nom).strip().upper()

    nom = unicodedata.normalize(
        "NFKD",
        nom,
    )

    nom = "".join(
        caractere
        for caractere in nom
        if not unicodedata.combining(caractere)
    )

    nom = re.sub(
        r"[^A-Z0-9]+",
        "_",
        nom,
    )

    return nom.strip("_")


def normaliser_code_commune(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
        .str.upper()
        .str.zfill(5)
    )


def pourcentage(numerateur, denominateur):
    numerateur = pd.to_numeric(
        numerateur,
        errors="coerce",
    )

    denominateur = pd.to_numeric(
        denominateur,
        errors="coerce",
    )

    denominateur = denominateur.where(
        denominateur.ne(0)
    )

    return (
        numerateur
        .div(denominateur)
        .mul(100)
    )


def taux_pour_mille(numerateur, denominateur):
    numerateur = pd.to_numeric(
        numerateur,
        errors="coerce",
    )

    denominateur = pd.to_numeric(
        denominateur,
        errors="coerce",
    )

    denominateur = denominateur.where(
        denominateur.ne(0)
    )

    return (
        numerateur
        .div(denominateur)
        .mul(1_000)
    )


def enregistrer_csv(table, fichier):
    table.to_csv(
        fichier,
        index=False,
        sep=",",
        encoding="utf-8-sig",
    )

    print("Fichier créé :", fichier)

In [4]:
#Extraire le fichier brut sur les emplois et l'activité de la population en IDF

noms_classeur_possibles = [
    "base_cc_emploi_pop_active_2023_xlsx.xlsx",
    "base_cc_emploi_pop_active_2023.xlsx",
]

FICHIER_EMPLOI = None

for nom in noms_classeur_possibles:
    candidat = DOSSIER_RAW_RP / nom

    if candidat.exists():
        FICHIER_EMPLOI = candidat
        break


# Si le XLSX n'est pas encore extrait,
# chercher l'archive officielle.
if FICHIER_EMPLOI is None:
    archives = sorted(
        DOSSIER_RAW_RP.glob(
            "*emploi*pop*active*2023*.zip"
        )
    )

    if archives:
        ARCHIVE_EMPLOI = archives[0]

        with ZipFile(ARCHIVE_EMPLOI) as archive:
            membres_xlsx = [
                membre
                for membre in archive.namelist()
                if membre.lower().endswith(".xlsx")
            ]

            if len(membres_xlsx) != 1:
                raise ValueError(
                    "L'archive doit contenir un seul classeur XLSX."
                )

            membre = membres_xlsx[0]

            FICHIER_EMPLOI = (
                DOSSIER_RAW_RP
                / Path(membre).name
            )

            if not FICHIER_EMPLOI.exists():
                with archive.open(membre) as source:
                    with open(
                        FICHIER_EMPLOI,
                        "wb",
                    ) as destination:
                        shutil.copyfileobj(
                            source,
                            destination,
                        )

                print(
                    "Classeur extrait :",
                    FICHIER_EMPLOI,
                )

if FICHIER_EMPLOI is None:
    raise FileNotFoundError(
        "Le classeur Emploi-Activité RP2023 est absent. "
        "Télécharge l'archive "
        "base_cc_emploi_pop_active_2023_xlsx.zip."
    )

print("Classeur utilisé :", FICHIER_EMPLOI)
print(
    "Taille :",
    round(
        FICHIER_EMPLOI.stat().st_size
        / 1_000_000,
        2,
    ),
    "Mo",
)

Classeur utilisé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\rp2023\base_cc_emploi_pop_active_2023_xlsx.xlsx
Taille : 83.15 Mo


In [7]:
if not FICHIER_PROFIL_J7.exists():
    raise FileNotFoundError(
        f"Profil J7 introuvable : {FICHIER_PROFIL_J7}"
    )

profil_j7 = pd.read_csv(
    FICHIER_PROFIL_J7,
    sep=";",
    encoding="utf-8-sig",
    dtype={
        "CODGEO": "string",
    },
    low_memory=False,
)
print(profil_j7.columns)

profil_j7.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j7.columns
]

if "CODGEO" not in profil_j7.columns:
    raise ValueError(
        "La colonne CODGEO est absente du profil J7."
    )

profil_j7["CODGEO"] = (
    normaliser_code_commune(
        profil_j7["CODGEO"]
    )
)

assert profil_j7["CODGEO"].notna().all()
assert profil_j7["CODGEO"].is_unique
assert profil_j7["CODGEO"].str.fullmatch(
    r"\d{5}"
).all()

codes_profil = set(
    profil_j7["CODGEO"]
)

print("Dimensions du profil J7 :", profil_j7.shape)
print("Communes :", len(profil_j7))

display(
    profil_j7.head()
)

Index(['CODGEO', 'NOM_COMMUNE', 'DEP', 'REG', 'POPULATION_2011',
       'POPULATION_2016', 'POPULATION_2022', 'POP_0_14_ANS_2022',
       'POP_15_29_ANS_2022', 'POP_30_44_ANS_2022',
       ...
       'FEMMES_NON_SCOL_DIPLOME_SUP', 'FEMMES_NON_SCOL_BAC_OU_PLUS',
       'PART_FEMMES_SANS_DIPLOME_CEP_PCT', 'PART_FEMMES_BAC_OU_PLUS_PCT',
       'PART_FEMMES_DIPLOME_SUP_PCT', 'PART_FEMMES_BAC5_PLUS_PCT',
       'MANQUANT_J7A', 'MANQUANT_J7B', 'MANQUANT_J7C',
       'NB_BLOCS_J7_MANQUANTS'],
      dtype='object', length=253)
Dimensions du profil J7 : (1266, 253)
Communes : 1266


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,POP_18_24,POP_25_39,POP_40_54,POP_55_64,POP_65_79,POP_80_PLUS,POP_FEMMES,POP_HOMMES,POP_F_0_2,POP_F_3_5,POP_F_6_10,POP_F_11_14,POP_F_15_17,POP_F_18_24,POP_F_25_39,POP_F_40_54,POP_F_55_64,POP_F_65_79,POP_F_80_PLUS,POP_H_0_2,POP_H_3_5,POP_H_6_10,POP_H_11_14,POP_H_15_17,POP_H_18_24,POP_H_25_39,POP_H_40_54,POP_H_55_64,POP_H_65_79,POP_H_80_PLUS,POP_RP2023,POP_0_17,POP_15_24,POP_15_39,POP_18_39,POP_65_PLUS,PART_FEMMES_PCT,PART_0_17_PCT,PART_15_24_PCT,PART_15_39_PCT,PART_18_39_PCT,PART_65_PLUS_PCT,NB_MENAGES,NB_MENAGES_1_PERSONNE,NB_MENAGES_HOMMES_SEULS,NB_MENAGES_FEMMES_SEULES,NB_MENAGES_AUTRES_SANS_FAMILLE,NB_MENAGES_AVEC_FAMILLES,NB_MENAGES_COUPLE_SANS_ENFANT,NB_MENAGES_COUPLE_AVEC_ENFANTS,NB_MENAGES_FAMILLE_MONOPARENTALE,POP_MENAGES,POP_15_PLUS,POP_MENAGES_15_24,POP_MENAGES_25_39,POP_MENAGES_40_54,POP_MENAGES_55_64,POP_MENAGES_65_79,POP_MENAGES_80_PLUS,POP_VIVANT_SEULE_15_24,POP_VIVANT_SEULE_25_39,POP_VIVANT_SEULE_40_54,POP_VIVANT_SEULE_55_64,POP_VIVANT_SEULE_65_79,POP_VIVANT_SEULE_80_PLUS,POP_VIVANT_EN_COUPLE_15_24,POP_VIVANT_EN_COUPLE_25_39,POP_VIVANT_EN_COUPLE_40_54,POP_VIVANT_EN_COUPLE_55_64,POP_VIVANT_EN_COUPLE_65_79,POP_VIVANT_EN_COUPLE_80_PLUS,POP_15P_MARIEE,POP_15P_PACSEE,POP_15P_CONCUBINAGE,POP_15P_VEUVE_VEUF,...,NB_MENAGES_REF_ARTISAN_COMMERCANT_CHEF,NB_MENAGES_REF_CADRE,NB_MENAGES_REF_PROF_INTERMEDIAIRE,NB_MENAGES_REF_EMPLOYE,NB_MENAGES_REF_OUVRIER,NB_MENAGES_REF_RETRAITE,NB_MENAGES_REF_AUTRE,NB_FAMILLES,NB_FAMILLES_COUPLE_AVEC_ENFANTS,NB_FAMILLES_MONOPARENTALES,NB_FAMILLES_MONO_HOMMES,NB_FAMILLES_MONO_FEMMES,NB_FAMILLES_COUPLE_SANS_ENFANT,NB_FAMILLES_0_ENFANT_MOINS25,NB_FAMILLES_1_ENFANT_MOINS25,NB_FAMILLES_2_ENFANTS_MOINS25,NB_FAMILLES_3_ENFANTS_MOINS25,NB_FAMILLES_4PLUS_ENFANTS_MOINS25,TAILLE_MOYENNE_MENAGE,PART_MENAGES_1_PERSONNE_PCT,PART_MENAGES_AVEC_FAMILLES_PCT,PART_MENAGES_COUPLE_SANS_ENFANT_PCT,PART_MENAGES_COUPLE_AVEC_ENFANTS_PCT,PART_MENAGES_FAMILLE_MONOPARENTALE_PCT,NB_FAMILLES_AVEC_ENFANTS_MOINS25,PART_FAMILLES_AVEC_ENFANTS_MOINS25_PCT,PART_FAMILLES_MONOPARENTALES_PCT,PART_FAMILLES_3PLUS_ENFANTS_MOINS25_PCT,PART_VIVANT_SEULE_15_24_PCT,PART_VIVANT_EN_COUPLE_15_24_PCT,PART_VIVANT_SEULE_25_39_PCT,PART_VIVANT_EN_COUPLE_25_39_PCT,PART_VIVANT_SEULE_40_54_PCT,PART_VIVANT_EN_COUPLE_40_54_PCT,PART_VIVANT_SEULE_55_64_PCT,PART_VIVANT_EN_COUPLE_55_64_PCT,PART_VIVANT_SEULE_65_79_PCT,PART_VIVANT_EN_COUPLE_65_79_PCT,PART_VIVANT_SEULE_80_PLUS_PCT,PART_VIVANT_EN_COUPLE_80_PLUS_PCT,POP_VIVANT_SEULE_15_PLUS,POP_VIVANT_EN_COUPLE_15_PLUS,PART_VIVANT_SEULE_15_PLUS_PCT,PART_VIVANT_EN_COUPLE_15_PLUS_PCT,PART_15P_MARIEE_PCT,PART_15P_PACSEE_PCT,PART_15P_CONCUBINAGE_PCT,PART_15P_VEUVE_VEUF_PCT,PART_15P_DIVORCEE_PCT,PART_15P_CELIBATAIRE_PCT,PART_MENAGES_REF_AGRICULTEUR_PCT,PART_MENAGES_REF_ARTISAN_COMMERCANT_CHEF_PCT,PART_MENAGES_REF_CADRE_PCT,PART_MENAGES_REF_PROF_INTERMEDIAIRE_PCT,PART_MENAGES_REF_EMPLOYE_PCT,PART_MENAGES_REF_OUVRIER_PCT,PART_MENAGES_REF_RETRAITE_PCT,PART_MENAGES_REF_AUTRE_PCT,POP_REFERENCE_

In [10]:
colonnes_source = {
    # Géographie
    "Code géographique": "CODGEO",
    "Libellé géographique": "LIBGEO_EMPLOI",

    # Population par âge
    "Pop 15-64 ans (princ)": "POPULATION_15_64",
    "Pop 15-24 ans (princ)": "POPULATION_15_24_ANS",
    "Pop 25-54 ans (princ)": "POPULATION_25_54_ANS",
    "Pop 55-64 ans (princ)": "POPULATION_55_64_ANS",

    # Population par sexe
    "Pop 15-64 ans Hommes (princ)": "POPULATION_15_64_HOMMES",
    "Pop 15-64 ans Femmes (princ)": "POPULATION_15_64_FEMMES",

    # Population active
    "Actifs 15-64 ans (princ)": "ACTIFS_15_64",
    "Actifs 15-24 ans (princ)": "ACTIFS_15_24",
    "Actifs 25-54 ans (princ)": "ACTIFS_25_54",
    "Actifs 55-64 ans (princ)": "ACTIFS_55_64",
    "Actifs 15-64 ans Hommes (princ)": "ACTIFS_15_64_HOMMES",
    "Actifs 15-64 ans Femmes (princ)": "ACTIFS_15_64_FEMMES",

    # Actifs occupés
    "Actifs occupés 15-64 ans (princ)": "ACTIFS_OCCUPES_15_64",
    "Actifs occupés 15-24 ans (princ)": "ACTIFS_OCCUPES_15_24",
    "Actifs occupés 25-54 ans (princ)": "ACTIFS_OCCUPES_25_54",
    "Actifs occupés 55-64 ans (princ)": "ACTIFS_OCCUPES_55_64",
    "Actifs occupés 15-64 ans Hommes (princ)": "ACTIFS_OCCUPES_15_64_HOMMES",
    "Actifs occupés 15-64 ans Femmes (princ)": "ACTIFS_OCCUPES_15_64_FEMMES",

    # Chômage
    "Chômeurs 15-64 ans (princ)": "CHOMEURS_15_64",
    "Chômeurs de 15-24 ans (princ)": "CHOMEURS_15_24",
    "Chômeurs de 25-54 ans (princ)": "CHOMEURS_25_54",
    "Chômeurs de 55-64 ans (princ)": "CHOMEURS_55_64",
    "Chômeurs 15-64 ans Hommes (princ)": "CHOMEURS_15_64_HOMMES",
    "Chômeurs 15-64 ans Femmes (princ)": "CHOMEURS_15_64_FEMMES",

    # Inactivité
    "Inactifs 15-64 ans (princ)": "INACTIFS_15_64",
    "Elèv. Etud. Stag. non rémunérés 15-64 ans (princ)": (
        "ELEVES_ETUDIANTS_STAGIAIRES_15_64"
    ),
    "Retraités Préretraités 15-64 ans (princ)": (
        "RETRAITES_PRERETRAITES_15_64"
    ),
    "Autres inactifs 15-64 ans (princ)": (
        "AUTRES_INACTIFS_15_64"
    ),

    # Actifs occupés résidents par CSP
    "Actifs occupés 15-64 ans (compl)": (
        "ACTIFS_OCCUPES_15_64_COMP"
    ),
    "Actifs occ 15-64 ans Agriculteurs exploitants (compl)": (
        "ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS"
    ),
    "Actifs occ 15-64 ans Artisans, Comm., Chefs entr. (compl)": (
        "ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS"
    ),
    "Actifs occ 15-64 ans Cadres Prof. intel. sup. (compl)": (
        "ACTIFS_OCCUPES_RESIDENTS_CADRES"
    ),
    "Actifs occ 15-64 ans Prof. intermédiaires (compl)": (
        "ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES"
    ),
    "Actifs occupés 15-64 ans Employés (compl)": (
        "ACTIFS_OCCUPES_RESIDENTS_EMPLOYES"
    ),
    "Actifs occupés 15-64 ans Ouvriers (compl)": (
        "ACTIFS_OCCUPES_RESIDENTS_OUVRIERS"
    ),

    # Emplois au lieu de travail
    "Emplois au LT (princ)": "EMPLOIS_LT_PRINC",
    "Actifs occupés 15 ans ou plus (princ)": (
        "ACTIFS_OCCUPES_RESIDENTS_15P"
    ),
    "Emploi des 15 ans ou plus au LT (princ)": (
        "EMPLOIS_15P_LT"
    ),

    # Statut et temps de travail
    "Emplois salariés au LT (princ)": "EMPLOIS_SALARIES_LT",
    "Emplois salariés femmes au LT (princ)": (
        "EMPLOIS_SALARIES_FEMMES_LT"
    ),
    "Emplois salariés TP au LT (princ)": (
        "EMPLOIS_SALARIES_TEMPS_PARTIEL_LT"
    ),
    "Emplois non-salariés au LT (princ)": (
        "EMPLOIS_NON_SALARIES_LT"
    ),
    "Emplois non-salariés femmes au LT (princ)": (
        "EMPLOIS_NON_SALARIES_FEMMES_LT"
    ),
    "Emplois non-salariés TP au LT (princ)": (
        "EMPLOIS_NON_SALARIES_TEMPS_PARTIEL_LT"
    ),

    # Exploitation complémentaire
    "Emplois au LT (compl)": "EMPLOIS_LT_COMP",
    "Emplois femmes au LT (compl)": "EMPLOIS_FEMMES_LT_COMP",

    # Emplois par CSP
    "Agriculteurs exploitants au LT (compl)": (
        "EMPLOIS_LT_AGRICULTEURS"
    ),
    "Artisans, Commerçants, Chefs entreprise au LT (compl)": (
        "EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS"
    ),
    "Cadres Prof. intel. sup. au LT (compl)": (
        "EMPLOIS_LT_CADRES"
    ),
    "Prof. intermédiaires au LT (compl)": (
        "EMPLOIS_LT_PROF_INTERMEDIAIRES"
    ),
    "Employés au LT (compl)": "EMPLOIS_LT_EMPLOYES",
    "Ouvriers au LT (compl)": "EMPLOIS_LT_OUVRIERS",

    # Emplois par secteur
    "Emplois au LT Agriculture (compl)": (
        "EMPLOIS_LT_AGRICULTURE"
    ),
    "Emplois au LT Industrie (compl)": (
        "EMPLOIS_LT_INDUSTRIE"
    ),
    "Emplois au LT Construction (compl)": (
        "EMPLOIS_LT_CONSTRUCTION"
    ),
    "Emplois au LT Commerce, Transports, Services divers (compl)": (
        "EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES"
    ),
    "Emplois au LT Adm publique, Enseignement, Santé, Act sociale (compl)": (
        "EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL"
    ),
}

print(
    "Nombre de colonnes sélectionnées :",
    len(colonnes_source),
)

Nombre de colonnes sélectionnées : 59


In [11]:
emploi_source = pd.read_excel(
    FICHIER_EMPLOI,
    sheet_name="COM_2023",
    usecols=list(colonnes_source),
    dtype={
        "Code géographique": "string",
    },
    engine="openpyxl",
)

emploi_source = emploi_source.rename(
    columns=colonnes_source
)

emploi_source["CODGEO"] = (
    normaliser_code_commune(
        emploi_source["CODGEO"]
    )
)

print(
    "Dimensions nationales après sélection :",
    emploi_source.shape,
)

display(
    emploi_source.head()
)

Dimensions nationales après sélection : (34858, 59)


,CODGEO,LIBGEO_EMPLOI,POPULATION_15_64,POPULATION_15_24_ANS,POPULATION_25_54_ANS,POPULATION_55_64_ANS,POPULATION_15_64_HOMMES,POPULATION_15_64_FEMMES,ACTIFS_15_64,ACTIFS_15_24,ACTIFS_25_54,ACTIFS_55_64,ACTIFS_15_64_HOMMES,ACTIFS_15_64_FEMMES,ACTIFS_OCCUPES_15_64,ACTIFS_OCCUPES_15_24,ACTIFS_OCCUPES_25_54,ACTIFS_OCCUPES_55_64,ACTIFS_OCCUPES_15_64_HOMMES,ACTIFS_OCCUPES_15_64_FEMMES,CHOMEURS_15_64,CHOMEURS_15_24,CHOMEURS_25_54,CHOMEURS_55_64,INACTIFS_15_64,ELEVES_ETUDIANTS_STAGIAIRES_15_64,RETRAITES_PRERETRAITES_15_64,AUTRES_INACTIFS_15_64,ACTIFS_OCCUPES_15_64_COMP,ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS,ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS,ACTIFS_OCCUPES_RESIDENTS_CADRES,ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES,ACTIFS_OCCUPES_RESIDENTS_EMPLOYES,ACTIFS_OCCUPES_RESIDENTS_OUVRIERS,EMPLOIS_LT_PRINC,ACTIFS_OCCUPES_RESIDENTS_15P,EMPLOIS_15P_LT,EMPLOIS_SALARIES_LT,EMPLOIS_SALARIES_FEMMES_LT,EMPLOIS_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_NON_SALARIES_LT,EMPLOIS_NON_SALARIES_FEMMES_LT,EMPLOIS_NON_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_LT_COMP,EMPLOIS_LT_AGRICULTEURS,EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS,EMPLOIS_LT_CADRES,EMPLOIS_LT_PROF_INTERMEDIAIRES,EMPLOIS_LT_EMPLOYES,EMPLOIS_LT_OUVRIERS,EMPLOIS_LT_AGRICULTURE,EMPLOIS_LT_INDUSTRIE,EMPLOIS_LT_CONSTRUCTION,EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES,EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL,EMPLOIS_FEMMES_LT_COMP,CHOMEURS_15_64_HOMMES,CHOMEURS_15_64_FEMMES
0,01001,L'Abergement-Clémenciat,525.59207,65.80238,327.86324,131.92646,262.27494,263.31714,431.37739,29.95146,319.89047,81.53546,224.20138,207.17601,403.38565,25.96181,300.91466,76.50919,213.21915,190.16650,27.99174,3.98965,18.97582,5.02627,94.21468,34.86410,41.32522,18.02536,442.07285,4.91369,39.24090,73.39378,98.55560,108.13828,117.83061,77.34773,408.42616,77.34773,46.32108,21.78348,11.87209,31.02665,11.00275,4.98985,78.48725,4.91369,24.60402,0.00000,9.91868,14.70725,24.34360,4.91369,14.77639,24.49177,9.67946,24.62593,39.33244,10.98223,17.00951
1,01002,L'Abergement-de-Varey,163.90656,24.35625,113.48694,26.06338,86.81272,77.09384,131.47620,4.95604,110.48284,16.03732,70.20250,61.27370,129.57305,4.95604,109.50687,15.11014,69.27532,60.29773,1.90314,0.00000,0.97597,0.92718,32.43037,17.47172,9.02474,5.93391,112.73972,24.26971,4.84145,19.80631,29.49162,14.83830,19.49233,25.72001,132.60283,25.72001,7.90852,5.00538,4.07820,17.81148,5.86920,4.91858,19.55401,14.58682,0.00000,4.96719,0.00000,0.00000,0.00000,14.58682,0.00000,0.00000,4.96719,0.00000,9.68290,0.92718,0.97597
2,01004,Ambérieu-en-Bugey,10034.54018,2093.28605,6117.08618,1824.16795,5071.15848,4963.38170,7804.66624,1090.19561,5552.61297,1161.85765,4062.01629,3742.64995,6787.64588,831.86111,4909.05594,1046.72883,3639.06878,3148.57709,1017.02036,258.33450,643.55703,115.12882,2229.87394,852.88753,482.32933,894.65708,6996.28095,20.35763,333.00081,1190.91912,1890.63482,1884.22560,1677.14296,8290.50492,6859.89560,8287.55278,7385.93194,3922.57706,1237.06146,904.57298,405.27792,160.15748,7955.11844,13.81700,440.95889,1153.18661,2266.92700,2467.89384,1612.33509,19.68901,607.35588,446.23187,3641.57505,3240.26663,4115.03418,422.94750,594.07285
3,01005,Ambérieux-en-Dombes,1222.27842,167.04194,783.14759,272.08890,608.01326,614.26517,994.70013,84.38124,749.94198,160.37690,508.76410,485.93603,931.29840,76.81704,706.79685,147.68452,485.73850,445.55990,63.40173,7.56420,43.14514,12.69239,227.57829,75.36568,85.07030,67.14232,934.26293,5.72737,94.97981,128.48764,205.23566,305.84584,193.98661,260.92658,943.78774,260.92658,169.21441,102.23654,42.09726,91.71217,40.27635,12.63832,324.92444,5.30645,68.62275,42.86866,94.16407,71.36992,42.59258,11.45090,60.24728,68.86006,89.87787,94.48833,142.29923,23.02560,40.37612
4,01006,Ambléon,72.63158,8.07018,40.35088,24.21053,42.36842,30.26316,51.44737,4.03509,36.31579,11.09649,29.25439,22.19298,46.40351,3.02632,33.28947,10.08772,26.22807,20.17544,5.04386,1.00877,3.02632,1.00877,21.18421,3.02632,12.10526,6.05263,67.95455,0.00000,0.00000,0.00000,10.45455,15.68182,41.8181

In [13]:
#Vérifier le classeur
with open(FICHIER_EMPLOI, "rb") as flux:
    signature = flux.read(4)

if signature != b"PK\x03\x04":
    raise ValueError(
        "Le fichier n'est pas un véritable classeur XLSX."
    )

classeur_emploi = pd.ExcelFile(
    FICHIER_EMPLOI,
    engine="openpyxl",
)

print(
    "Onglets :",
    classeur_emploi.sheet_names,
)

assert "COM_2023" in classeur_emploi.sheet_names

print("Classeur valide ✅")



Onglets : ['Métadonnées', 'COM_2023', 'ARM_2023', 'COM_2017', 'ARM_2017', 'COM_2012', 'ARM_2012', 'Documentation']
Classeur valide ✅


In [14]:
#Vérifier les intitulés de colonnes

entete_emploi = pd.read_excel(
    FICHIER_EMPLOI,
    sheet_name="COM_2023",
    nrows=0,
    engine="openpyxl",
)

colonnes_absentes = (
    set(colonnes_source)
    - set(entete_emploi.columns)
)

if colonnes_absentes:
    raise ValueError(
        "Colonnes absentes du classeur emploi : "
        f"{sorted(colonnes_absentes)}"
    )

print("Toutes les colonnes nécessaires sont présentes ✅")

Toutes les colonnes nécessaires sont présentes ✅


In [15]:
#Charger uniquement les colonnes utiles

emploi_source = pd.read_excel(
    FICHIER_EMPLOI,
    sheet_name="COM_2023",
    usecols=list(colonnes_source),
    dtype={
        "Code géographique": "string",
    },
    engine="openpyxl",
)

emploi_source = emploi_source.rename(
    columns=colonnes_source
)

emploi_source["CODGEO"] = (
    normaliser_code_commune(
        emploi_source["CODGEO"]
    )
)

print(
    "Dimensions nationales après sélection :",
    emploi_source.shape,
)

display(
    emploi_source.head()
)

Dimensions nationales après sélection : (34858, 59)


,CODGEO,LIBGEO_EMPLOI,POPULATION_15_64,POPULATION_15_24_ANS,POPULATION_25_54_ANS,POPULATION_55_64_ANS,POPULATION_15_64_HOMMES,POPULATION_15_64_FEMMES,ACTIFS_15_64,ACTIFS_15_24,ACTIFS_25_54,ACTIFS_55_64,ACTIFS_15_64_HOMMES,ACTIFS_15_64_FEMMES,ACTIFS_OCCUPES_15_64,ACTIFS_OCCUPES_15_24,ACTIFS_OCCUPES_25_54,ACTIFS_OCCUPES_55_64,ACTIFS_OCCUPES_15_64_HOMMES,ACTIFS_OCCUPES_15_64_FEMMES,CHOMEURS_15_64,CHOMEURS_15_24,CHOMEURS_25_54,CHOMEURS_55_64,INACTIFS_15_64,ELEVES_ETUDIANTS_STAGIAIRES_15_64,RETRAITES_PRERETRAITES_15_64,AUTRES_INACTIFS_15_64,ACTIFS_OCCUPES_15_64_COMP,ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS,ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS,ACTIFS_OCCUPES_RESIDENTS_CADRES,ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES,ACTIFS_OCCUPES_RESIDENTS_EMPLOYES,ACTIFS_OCCUPES_RESIDENTS_OUVRIERS,EMPLOIS_LT_PRINC,ACTIFS_OCCUPES_RESIDENTS_15P,EMPLOIS_15P_LT,EMPLOIS_SALARIES_LT,EMPLOIS_SALARIES_FEMMES_LT,EMPLOIS_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_NON_SALARIES_LT,EMPLOIS_NON_SALARIES_FEMMES_LT,EMPLOIS_NON_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_LT_COMP,EMPLOIS_LT_AGRICULTEURS,EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS,EMPLOIS_LT_CADRES,EMPLOIS_LT_PROF_INTERMEDIAIRES,EMPLOIS_LT_EMPLOYES,EMPLOIS_LT_OUVRIERS,EMPLOIS_LT_AGRICULTURE,EMPLOIS_LT_INDUSTRIE,EMPLOIS_LT_CONSTRUCTION,EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES,EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL,EMPLOIS_FEMMES_LT_COMP,CHOMEURS_15_64_HOMMES,CHOMEURS_15_64_FEMMES
0,01001,L'Abergement-Clémenciat,525.59207,65.80238,327.86324,131.92646,262.27494,263.31714,431.37739,29.95146,319.89047,81.53546,224.20138,207.17601,403.38565,25.96181,300.91466,76.50919,213.21915,190.16650,27.99174,3.98965,18.97582,5.02627,94.21468,34.86410,41.32522,18.02536,442.07285,4.91369,39.24090,73.39378,98.55560,108.13828,117.83061,77.34773,408.42616,77.34773,46.32108,21.78348,11.87209,31.02665,11.00275,4.98985,78.48725,4.91369,24.60402,0.00000,9.91868,14.70725,24.34360,4.91369,14.77639,24.49177,9.67946,24.62593,39.33244,10.98223,17.00951
1,01002,L'Abergement-de-Varey,163.90656,24.35625,113.48694,26.06338,86.81272,77.09384,131.47620,4.95604,110.48284,16.03732,70.20250,61.27370,129.57305,4.95604,109.50687,15.11014,69.27532,60.29773,1.90314,0.00000,0.97597,0.92718,32.43037,17.47172,9.02474,5.93391,112.73972,24.26971,4.84145,19.80631,29.49162,14.83830,19.49233,25.72001,132.60283,25.72001,7.90852,5.00538,4.07820,17.81148,5.86920,4.91858,19.55401,14.58682,0.00000,4.96719,0.00000,0.00000,0.00000,14.58682,0.00000,0.00000,4.96719,0.00000,9.68290,0.92718,0.97597
2,01004,Ambérieu-en-Bugey,10034.54018,2093.28605,6117.08618,1824.16795,5071.15848,4963.38170,7804.66624,1090.19561,5552.61297,1161.85765,4062.01629,3742.64995,6787.64588,831.86111,4909.05594,1046.72883,3639.06878,3148.57709,1017.02036,258.33450,643.55703,115.12882,2229.87394,852.88753,482.32933,894.65708,6996.28095,20.35763,333.00081,1190.91912,1890.63482,1884.22560,1677.14296,8290.50492,6859.89560,8287.55278,7385.93194,3922.57706,1237.06146,904.57298,405.27792,160.15748,7955.11844,13.81700,440.95889,1153.18661,2266.92700,2467.89384,1612.33509,19.68901,607.35588,446.23187,3641.57505,3240.26663,4115.03418,422.94750,594.07285
3,01005,Ambérieux-en-Dombes,1222.27842,167.04194,783.14759,272.08890,608.01326,614.26517,994.70013,84.38124,749.94198,160.37690,508.76410,485.93603,931.29840,76.81704,706.79685,147.68452,485.73850,445.55990,63.40173,7.56420,43.14514,12.69239,227.57829,75.36568,85.07030,67.14232,934.26293,5.72737,94.97981,128.48764,205.23566,305.84584,193.98661,260.92658,943.78774,260.92658,169.21441,102.23654,42.09726,91.71217,40.27635,12.63832,324.92444,5.30645,68.62275,42.86866,94.16407,71.36992,42.59258,11.45090,60.24728,68.86006,89.87787,94.48833,142.29923,23.02560,40.37612
4,01006,Ambléon,72.63158,8.07018,40.35088,24.21053,42.36842,30.26316,51.44737,4.03509,36.31579,11.09649,29.25439,22.19298,46.40351,3.02632,33.28947,10.08772,26.22807,20.17544,5.04386,1.00877,3.02632,1.00877,21.18421,3.02632,12.10526,6.05263,67.95455,0.00000,0.00000,0.00000,10.45455,15.68182,41.8181

In [20]:
#Filtrer les communes de l'IDF

codes_source = set(
    emploi_source["CODGEO"]
    .dropna()
)

codes_absents = sorted(
    codes_profil - codes_source
)

if codes_absents:
    raise ValueError(
        "Codes du profil J7 absents de la source emploi : "
        f"{codes_absents}"
    )

emploi_idf = emploi_source[
    emploi_source["CODGEO"].isin(
        codes_profil
    )
].copy()

del emploi_source

emploi_idf = (
    emploi_idf
    .sort_values("CODGEO")
    .reset_index(drop=True)
)

colonnes_numeriques = [
    colonne
    for colonne in emploi_idf.columns
    if colonne not in [
        "CODGEO",
        "LIBGEO_EMPLOI",
    ]
]

emploi_idf[
    colonnes_numeriques
] = emploi_idf[
    colonnes_numeriques
].apply(
    pd.to_numeric,
    errors="coerce",
)

valeurs_manquantes = (
    emploi_idf[
        colonnes_numeriques
    ]
    .isna()
    .sum()
)

valeurs_manquantes = valeurs_manquantes[
    valeurs_manquantes.gt(0)
]

if not valeurs_manquantes.empty:
    display(
        valeurs_manquantes
        .rename("NB_VALEURS_MANQUANTES")
        .reset_index()
    )

    raise ValueError(
        "Des valeurs numériques sont manquantes "
        "dans la source emploi."
    )

assert len(emploi_idf) == len(profil_j7)
assert emploi_idf["CODGEO"].is_unique
assert "75056" in emploi_idf["CODGEO"].values
assert "93066" in emploi_idf["CODGEO"].values
assert "93059" not in emploi_idf["CODGEO"].values

print("Communes IDF :", len(emploi_idf))
print("Filtrage réussi ✅")

Communes IDF : 1266
Filtrage réussi ✅


In [22]:
emploi_idf.columns

Index(['CODGEO', 'LIBGEO_EMPLOI', 'POPULATION_15_64', 'POPULATION_15_24_ANS',
       'POPULATION_25_54_ANS', 'POPULATION_55_64_ANS',
       'POPULATION_15_64_HOMMES', 'POPULATION_15_64_FEMMES', 'ACTIFS_15_64',
       'ACTIFS_15_24', 'ACTIFS_25_54', 'ACTIFS_55_64', 'ACTIFS_15_64_HOMMES',
       'ACTIFS_15_64_FEMMES', 'ACTIFS_OCCUPES_15_64', 'ACTIFS_OCCUPES_15_24',
       'ACTIFS_OCCUPES_25_54', 'ACTIFS_OCCUPES_55_64',
       'ACTIFS_OCCUPES_15_64_HOMMES', 'ACTIFS_OCCUPES_15_64_FEMMES',
       'CHOMEURS_15_64', 'CHOMEURS_15_24', 'CHOMEURS_25_54', 'CHOMEURS_55_64',
       'INACTIFS_15_64', 'ELEVES_ETUDIANTS_STAGIAIRES_15_64',
       'RETRAITES_PRERETRAITES_15_64', 'AUTRES_INACTIFS_15_64',
       'ACTIFS_OCCUPES_15_64_COMP', 'ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS',
       'ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS',
       'ACTIFS_OCCUPES_RESIDENTS_CADRES',
       'ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES',
       'ACTIFS_OCCUPES_RESIDENTS_EMPLOYES',
       'ACTIFS_OCCUPES_RE

In [26]:
#Calculer les taux d'activité, d'emploi et de chômage
tranches_activite = [
    "15_64",
    "15_24",
    "25_54",
    "55_64",
]

for tranche in tranches_activite:
    emploi_idf[
        f"TAUX_ACTIVITE_{tranche}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"ACTIFS_{tranche}"
        ],
        emploi_idf[
            (
                "POPULATION_15_64"
                if tranche == "15_64"
                else f"POPULATION_{tranche}_ANS"
            )
        ],
    )

    emploi_idf[
        f"TAUX_EMPLOI_{tranche}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"ACTIFS_OCCUPES_{tranche}"
        ],
        emploi_idf[
            (
                "POPULATION_15_64"
                if tranche == "15_64"
                else f"POPULATION_{tranche}_ANS"
            )
        ],
    )

    emploi_idf[
        f"TAUX_CHOMAGE_RP_{tranche}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"CHOMEURS_{tranche}"
        ],
        emploi_idf[
            f"ACTIFS_{tranche}"
        ],
    )

In [27]:
#Calculer les indicateurs par sexe
for sexe in [
    "HOMMES",
    "FEMMES",
]:
    emploi_idf[
        f"TAUX_ACTIVITE_15_64_{sexe}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"ACTIFS_15_64_{sexe}"
        ],
        emploi_idf[
            f"POPULATION_15_64_{sexe}"
        ],
    )

    emploi_idf[
        f"TAUX_EMPLOI_15_64_{sexe}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"ACTIFS_OCCUPES_15_64_{sexe}"
        ],
        emploi_idf[
            f"POPULATION_15_64_{sexe}"
        ],
    )

    emploi_idf[
        f"TAUX_CHOMAGE_RP_15_64_{sexe}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"CHOMEURS_15_64_{sexe}"
        ],
        emploi_idf[
            f"ACTIFS_15_64_{sexe}"
        ],
    )

emploi_idf[
    "ECART_TAUX_EMPLOI_FEMMES_HOMMES_POINTS"
] = (
    emploi_idf[
        "TAUX_EMPLOI_15_64_FEMMES_PCT"
    ]
    - emploi_idf[
        "TAUX_EMPLOI_15_64_HOMMES_PCT"
    ]
)

In [28]:
#Mesurer l'inactivité
emploi_idf[
    "PART_INACTIFS_15_64_PCT"
] = pourcentage(
    emploi_idf["INACTIFS_15_64"],
    emploi_idf["POPULATION_15_64"],
)

emploi_idf[
    "PART_ELEVES_ETUDIANTS_STAGIAIRES_15_64_PCT"
] = pourcentage(
    emploi_idf[
        "ELEVES_ETUDIANTS_STAGIAIRES_15_64"
    ],
    emploi_idf["POPULATION_15_64"],
)

emploi_idf[
    "PART_RETRAITES_PRERETRAITES_15_64_PCT"
] = pourcentage(
    emploi_idf[
        "RETRAITES_PRERETRAITES_15_64"
    ],
    emploi_idf["POPULATION_15_64"],
)

emploi_idf[
    "PART_AUTRES_INACTIFS_15_64_PCT"
] = pourcentage(
    emploi_idf[
        "AUTRES_INACTIFS_15_64"
    ],
    emploi_idf["POPULATION_15_64"],
)

In [29]:
#Calculer la concentration d'emploi

emploi_idf[
    "INDICE_CONCENTRATION_EMPLOI"
] = pourcentage(
    emploi_idf["EMPLOIS_15P_LT"],
    emploi_idf[
        "ACTIFS_OCCUPES_RESIDENTS_15P"
    ],
)

emploi_idf[
    "ECART_EMPLOIS_LT_ACTIFS_RESIDENTS"
] = (
    emploi_idf["EMPLOIS_15P_LT"]
    - emploi_idf[
        "ACTIFS_OCCUPES_RESIDENTS_15P"
    ]
)

In [30]:
#Statut et temps de travail des emplois
emploi_idf[
    "EMPLOIS_TEMPS_PARTIEL_LT"
] = (
    emploi_idf[
        "EMPLOIS_SALARIES_TEMPS_PARTIEL_LT"
    ]
    + emploi_idf[
        "EMPLOIS_NON_SALARIES_TEMPS_PARTIEL_LT"
    ]
)

emploi_idf[
    "PART_EMPLOIS_SALARIES_LT_PCT"
] = pourcentage(
    emploi_idf["EMPLOIS_SALARIES_LT"],
    emploi_idf["EMPLOIS_LT_PRINC"],
)

emploi_idf[
    "PART_EMPLOIS_NON_SALARIES_LT_PCT"
] = pourcentage(
    emploi_idf["EMPLOIS_NON_SALARIES_LT"],
    emploi_idf["EMPLOIS_LT_PRINC"],
)

emploi_idf[
    "PART_EMPLOIS_TEMPS_PARTIEL_LT_PCT"
] = pourcentage(
    emploi_idf["EMPLOIS_TEMPS_PARTIEL_LT"],
    emploi_idf["EMPLOIS_LT_PRINC"],
)

emploi_idf[
    "PART_EMPLOIS_FEMMES_LT_PCT"
] = pourcentage(
    emploi_idf["EMPLOIS_FEMMES_LT_COMP"],
    emploi_idf["EMPLOIS_LT_COMP"],
)

In [31]:
#Part des catégories socio-professionnelles
categories_pcs = [
    "AGRICULTEURS",
    "ARTISANS_COMMERCANTS_CHEFS",
    "CADRES",
    "PROF_INTERMEDIAIRES",
    "EMPLOYES",
    "OUVRIERS",
]

for categorie in categories_pcs:
    emploi_idf[
        f"PART_EMPLOIS_LT_{categorie}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"EMPLOIS_LT_{categorie}"
        ],
        emploi_idf["EMPLOIS_LT_COMP"],
    )

    emploi_idf[
        f"PART_ACTIFS_OCCUPES_RESIDENTS_{categorie}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"ACTIFS_OCCUPES_RESIDENTS_{categorie}"
        ],
        emploi_idf[
            "ACTIFS_OCCUPES_15_64_COMP"
        ],
    )

In [34]:
#Part des secteurs d'activité
secteurs_activite = [
    "AGRICULTURE",
    "INDUSTRIE",
    "CONSTRUCTION",
    "COMMERCE_TRANSPORTS_SERVICES",
    "ADMIN_ENSEIGNEMENT_SANTE_SOCIAL",
]

for secteur in secteurs_activite:
    emploi_idf[
        f"PART_EMPLOIS_LT_{secteur}_PCT"
    ] = pourcentage(
        emploi_idf[
            f"EMPLOIS_LT_{secteur}"
        ],
        emploi_idf["EMPLOIS_LT_COMP"],
    )

emploi_idf[
    "EMPLOIS_TERTIAIRES_LT"
] = (
    emploi_idf[
        "EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES"
    ]
    + emploi_idf[
        "EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL"
    ]
)

emploi_idf[
    "PART_EMPLOIS_TERTIAIRES_LT_PCT"
] = pourcentage(
    emploi_idf["EMPLOIS_TERTIAIRES_LT"],
    emploi_idf["EMPLOIS_LT_COMP"],
)

In [35]:
emploi_idf

,CODGEO,LIBGEO_EMPLOI,POPULATION_15_64,POPULATION_15_24_ANS,POPULATION_25_54_ANS,POPULATION_55_64_ANS,POPULATION_15_64_HOMMES,POPULATION_15_64_FEMMES,ACTIFS_15_64,ACTIFS_15_24,ACTIFS_25_54,ACTIFS_55_64,ACTIFS_15_64_HOMMES,ACTIFS_15_64_FEMMES,ACTIFS_OCCUPES_15_64,ACTIFS_OCCUPES_15_24,ACTIFS_OCCUPES_25_54,ACTIFS_OCCUPES_55_64,ACTIFS_OCCUPES_15_64_HOMMES,ACTIFS_OCCUPES_15_64_FEMMES,CHOMEURS_15_64,CHOMEURS_15_24,CHOMEURS_25_54,CHOMEURS_55_64,INACTIFS_15_64,ELEVES_ETUDIANTS_STAGIAIRES_15_64,RETRAITES_PRERETRAITES_15_64,AUTRES_INACTIFS_15_64,ACTIFS_OCCUPES_15_64_COMP,ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS,ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS,ACTIFS_OCCUPES_RESIDENTS_CADRES,ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES,ACTIFS_OCCUPES_RESIDENTS_EMPLOYES,ACTIFS_OCCUPES_RESIDENTS_OUVRIERS,EMPLOIS_LT_PRINC,ACTIFS_OCCUPES_RESIDENTS_15P,EMPLOIS_15P_LT,EMPLOIS_SALARIES_LT,EMPLOIS_SALARIES_FEMMES_LT,EMPLOIS_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_NON_SALARIES_LT,EMPLOIS_NON_SALARIES_FEMMES_LT,EMPLOIS_NON_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_LT_COMP,EMPLOIS_LT_AGRICULTEURS,EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS,EMPLOIS_LT_CADRES,EMPLOIS_LT_PROF_INTERMEDIAIRES,EMPLOIS_LT_EMPLOYES,EMPLOIS_LT_OUVRIERS,EMPLOIS_LT_AGRICULTURE,EMPLOIS_LT_INDUSTRIE,EMPLOIS_LT_CONSTRUCTION,EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES,EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL,EMPLOIS_FEMMES_LT_COMP,CHOMEURS_15_64_HOMMES,CHOMEURS_15_64_FEMMES,TAUX_ACTIVITE_15_64_PCT,TAUX_EMPLOI_15_64_PCT,TAUX_CHOMAGE_RP_15_64_PCT,TAUX_ACTIVITE_15_24_PCT,TAUX_EMPLOI_15_24_PCT,TAUX_CHOMAGE_RP_15_24_PCT,TAUX_ACTIVITE_25_54_PCT,TAUX_EMPLOI_25_54_PCT,TAUX_CHOMAGE_RP_25_54_PCT,TAUX_ACTIVITE_55_64_PCT,TAUX_EMPLOI_55_64_PCT,TAUX_CHOMAGE_RP_55_64_PCT,TAUX_ACTIVITE_15_64_HOMMES_PCT,TAUX_EMPLOI_15_64_HOMMES_PCT,TAUX_CHOMAGE_RP_15_64_HOMMES_PCT,TAUX_ACTIVITE_15_64_FEMMES_PCT,TAUX_EMPLOI_15_64_FEMMES_PCT,TAUX_CHOMAGE_RP_15_64_FEMMES_PCT,ECART_TAUX_EMPLOI_FEMMES_HOMMES_POINTS,PART_INACTIFS_15_64_PCT,PART_ELEVES_ETUDIANTS_STAGIAIRES_15_64_PCT,PART_RETRAITES_PRERETRAITES_15_64_PCT,PART_AUTRES_INACTIFS_15_64_PCT,INDICE_CONCENTRATION_EMPLOI,ECART_EMPLOIS_LT_ACTIFS_RESIDENTS,EMPLOIS_TEMPS_PARTIEL_LT,PART_EMPLOIS_SALARIES_LT_PCT,PART_EMPLOIS_NON_SALARIES_LT_PCT,PART_EMPLOIS_TEMPS_PARTIEL_LT_PCT,PART_EMPLOIS_FEMMES_LT_PCT,PART_EMPLOIS_LT_AGRICULTEURS_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS_PCT,PART_EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS_PCT,PART_EMPLOIS_LT_CADRES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_CADRES_PCT,PART_EMPLOIS_LT_PROF_INTERMEDIAIRES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES_PCT,PART_EMPLOIS_LT_EMPLOYES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_EMPLOYES_PCT,PART_EMPLOIS_LT_OUVRIERS_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_OUVRIERS_PCT,PART_EMPLOIS_LT_AGRICULTURE_PCT,PART_EMPLOIS_LT_INDUSTRIE_PCT,PART_EMPLOIS_LT_CONSTRUCTION_PCT,PART_EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES_PCT,PART_EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL_PCT,EMPLOIS_TERTIAIRES_LT,PART_EMPLOIS_TERTIAIRES_LT_PCT
0,75056,Paris,1.463510e+06,295829.88697,930993.83316,236685.88601,698635.98977,764873.61637,1.152923e+06,114259.12193,857233.20755,181431.13939,564280.93601,588642.53287,1.032139e+06,93764.64809,775360.23802,163014.57520,506402.90475,525736.55655,120784.00757,20494.47384,81872.96953,18416.56420,310586.13727,186207.98165,32286.97587,92091.17974,1.039014e+06,295.44487,56194.67045,545936.19922,215963.67875,165750.66360,54873.51561,1.811255e+06,1.070897e+06,1.810931e+06,1.551134e+06,799334.01366,202430.18405,260121.10924,105157.07614,51380.91482,1.811126e+06,456.37444,94252.18789,756643.81241,419375.46508,388506.69315,151891.56650,914.36141,66166.60678,58704.78231,1.261996e+06,423344.24287,906630.74953,57878.03125,62905.97631,78.777991,70.524953,10.476325,38.623252,31.695462,17.936838,92.077217,83.283069,9.550840,76.654820,68.873805,10.150718,80.768948,72.484514,10.256953,76.959451,68.735088,10.686618,-3.749426,21.222009,12.723386,2.206134,6.292489,169.104215,74003

In [37]:
#Contrôler les égalités comptables
def ecart_max_composantes(
    table,
    colonne_total,
    colonnes_composantes,
):
    ecart = (
        table[colonne_total]
        - table[
            colonnes_composantes
        ].sum(axis=1)
    ).abs()

    return ecart.max()


controles_additivite = {
    "Population par âge": ecart_max_composantes(
        emploi_idf,
        "POPULATION_15_64",
        [
            "POPULATION_15_24_ANS",
            "POPULATION_25_54_ANS",
            "POPULATION_55_64_ANS",
        ],
    ),

    "Actifs par âge": ecart_max_composantes(
        emploi_idf,
        "ACTIFS_15_64",
        [
            "ACTIFS_15_24",
            "ACTIFS_25_54",
            "ACTIFS_55_64",
        ],
    ),

    "Actifs occupés par âge": ecart_max_composantes(
        emploi_idf,
        "ACTIFS_OCCUPES_15_64",
        [
            "ACTIFS_OCCUPES_15_24",
            "ACTIFS_OCCUPES_25_54",
            "ACTIFS_OCCUPES_55_64",
        ],
    ),

    "Chômeurs par âge": ecart_max_composantes(
        emploi_idf,
        "CHOMEURS_15_64",
        [
            "CHOMEURS_15_24",
            "CHOMEURS_25_54",
            "CHOMEURS_55_64",
        ],
    ),

    "Actifs occupés plus chômeurs": (
        emploi_idf["ACTIFS_15_64"]
        - emploi_idf[
            [
                "ACTIFS_OCCUPES_15_64",
                "CHOMEURS_15_64",
            ]
        ].sum(axis=1)
    ).abs().max(),

    "Actifs plus inactifs": (
        emploi_idf["POPULATION_15_64"]
        - emploi_idf[
            [
                "ACTIFS_15_64",
                "INACTIFS_15_64",
            ]
        ].sum(axis=1)
    ).abs().max(),

    "Composantes des inactifs": (
        emploi_idf["INACTIFS_15_64"]
        - emploi_idf[
            [
                "ELEVES_ETUDIANTS_STAGIAIRES_15_64",
                "RETRAITES_PRERETRAITES_15_64",
                "AUTRES_INACTIFS_15_64",
            ]
        ].sum(axis=1)
    ).abs().max(),

    "Emplois salariés plus non-salariés": (
        emploi_idf["EMPLOIS_LT_PRINC"]
        - emploi_idf[
            [
                "EMPLOIS_SALARIES_LT",
                "EMPLOIS_NON_SALARIES_LT",
            ]
        ].sum(axis=1)
    ).abs().max(),

    "Emplois par CSP": (
        emploi_idf["EMPLOIS_LT_COMP"]
        - emploi_idf[
            [
                "EMPLOIS_LT_AGRICULTEURS",
                "EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS",
                "EMPLOIS_LT_CADRES",
                "EMPLOIS_LT_PROF_INTERMEDIAIRES",
                "EMPLOIS_LT_EMPLOYES",
                "EMPLOIS_LT_OUVRIERS",
            ]
        ].sum(axis=1)
    ).abs().max(),

    "Emplois par secteur": (
        emploi_idf["EMPLOIS_LT_COMP"]
        - emploi_idf[
            [
                "EMPLOIS_LT_AGRICULTURE",
                "EMPLOIS_LT_INDUSTRIE",
                "EMPLOIS_LT_CONSTRUCTION",
                "EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES",
                "EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL",
            ]
        ].sum(axis=1)
    ).abs().max(),
}

controle_j8 = pd.DataFrame(
    {
        "CONTROLE": controles_additivite.keys(),
        "ECART_MAXIMAL": controles_additivite.values(),
    }
)

display(controle_j8)

assert (
    controle_j8["ECART_MAXIMAL"]
    < 0.001
).all()

print("Égalités comptables validées ✅")

,CONTROLE,ECART_MAXIMAL
0,Population par âge,0.00001
1,Actifs par âge,0.00001
2,Actifs occupés par âge,0.00001
3,Chômeurs par âge,0.00001
4,Actifs occupés plus chômeurs,0.00001
5,Actifs plus inactifs,0.00001
6,Composantes des inactifs,0.00001
7,Emplois salariés plus non-salariés,0.00001
8,Emplois par CSP,0.00002
9,Emplois par secteur,0.00002


Égalités comptables validées ✅


In [38]:
#Enregistrer la table intermédiaire

indicateurs_j8 = (
    emploi_idf
    .sort_values("CODGEO")
    .reset_index(drop=True)
)

enregistrer_csv(
    indicateurs_j8,
    FICHIER_INTERIM_J8,
)

enregistrer_csv(
    controle_j8,
    FICHIER_CONTROLE_J8,
)

Fichier créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j8_emploi_activite_idf_2023.csv
Fichier créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j8_controle_emploi_activite.csv


In [39]:
#Préparer la jointure à la table du profil J7

table_jointure_j8 = indicateurs_j8.drop(
    columns=[
        "LIBGEO_EMPLOI",
    ]
)

colonnes_en_conflit = (
    set(profil_j7.columns)
    & set(table_jointure_j8.columns)
) - {
    "CODGEO"
}

if colonnes_en_conflit:
    raise ValueError(
        "Colonnes J8 déjà présentes dans le profil J7 : "
        f"{sorted(colonnes_en_conflit)}"
    )

codes_j8 = set(
    table_jointure_j8["CODGEO"]
)

assert codes_j8 == codes_profil

print("Table J8 prête pour la jointure ✅")

Table J8 prête pour la jointure ✅


In [40]:
#Joindre à la table profil J7

profil_j8 = profil_j7.merge(
    table_jointure_j8,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

assert len(profil_j8) == len(profil_j7)
assert profil_j8["CODGEO"].is_unique

print("Communes avant jointure :", len(profil_j7))
print("Communes après jointure :", len(profil_j8))
print("Jointure réussie ✅")

Communes avant jointure : 1266
Communes après jointure : 1266
Jointure réussie ✅


In [41]:
#Calculer les indicateurs commerciaux

if "POP_RP2023" not in profil_j8.columns:
    raise ValueError(
        "POP_RP2023 est absente du profil J7."
    )

profil_j8[
    "EMPLOIS_15P_LT_POUR_1000_HAB"
] = taux_pour_mille(
    profil_j8["EMPLOIS_15P_LT"],
    profil_j8["POP_RP2023"],
)

densite_emplois_idf = (
    profil_j8["EMPLOIS_15P_LT"].sum()
    / profil_j8["POP_RP2023"].sum()
    * 1_000
)

profil_j8[
    "INDICE_DENSITE_EMPLOIS_IDF_BASE100"
] = (
    profil_j8[
        "EMPLOIS_15P_LT_POUR_1000_HAB"
    ]
    / densite_emplois_idf
    * 100
)

print(
    "Densité régionale :",
    round(densite_emplois_idf, 2),
    "emplois pour 1 000 habitants",
)

Densité régionale : 483.68 emplois pour 1 000 habitants


In [43]:
profil_j8

,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,POP_18_24,POP_25_39,POP_40_54,POP_55_64,POP_65_79,POP_80_PLUS,POP_FEMMES,POP_HOMMES,POP_F_0_2,POP_F_3_5,POP_F_6_10,POP_F_11_14,POP_F_15_17,POP_F_18_24,POP_F_25_39,POP_F_40_54,POP_F_55_64,POP_F_65_79,POP_F_80_PLUS,POP_H_0_2,POP_H_3_5,POP_H_6_10,POP_H_11_14,POP_H_15_17,POP_H_18_24,POP_H_25_39,POP_H_40_54,POP_H_55_64,POP_H_65_79,POP_H_80_PLUS,POP_RP2023,POP_0_17,POP_15_24,POP_15_39,POP_18_39,POP_65_PLUS,PART_FEMMES_PCT,PART_0_17_PCT,PART_15_24_PCT,PART_15_39_PCT,PART_18_39_PCT,PART_65_PLUS_PCT,NB_MENAGES,NB_MENAGES_1_PERSONNE,NB_MENAGES_HOMMES_SEULS,NB_MENAGES_FEMMES_SEULES,NB_MENAGES_AUTRES_SANS_FAMILLE,NB_MENAGES_AVEC_FAMILLES,NB_MENAGES_COUPLE_SANS_ENFANT,NB_MENAGES_COUPLE_AVEC_ENFANTS,NB_MENAGES_FAMILLE_MONOPARENTALE,POP_MENAGES,POP_15_PLUS,POP_MENAGES_15_24,POP_MENAGES_25_39,POP_MENAGES_40_54,POP_MENAGES_55_64,POP_MENAGES_65_79,POP_MENAGES_80_PLUS,POP_VIVANT_SEULE_15_24,POP_VIVANT_SEULE_25_39,POP_VIVANT_SEULE_40_54,POP_VIVANT_SEULE_55_64,POP_VIVANT_SEULE_65_79,POP_VIVANT_SEULE_80_PLUS,POP_VIVANT_EN_COUPLE_15_24,POP_VIVANT_EN_COUPLE_25_39,POP_VIVANT_EN_COUPLE_40_54,POP_VIVANT_EN_COUPLE_55_64,POP_VIVANT_EN_COUPLE_65_79,POP_VIVANT_EN_COUPLE_80_PLUS,POP_15P_MARIEE,POP_15P_PACSEE,POP_15P_CONCUBINAGE,POP_15P_VEUVE_VEUF,...,PART_POP_BAC5_PLUS_PCT,HOMMES_NON_SCOL_DIPLOME_SUP,HOMMES_NON_SCOL_BAC_OU_PLUS,PART_HOMMES_SANS_DIPLOME_CEP_PCT,PART_HOMMES_BAC_OU_PLUS_PCT,PART_HOMMES_DIPLOME_SUP_PCT,PART_HOMMES_BAC5_PLUS_PCT,FEMMES_NON_SCOL_DIPLOME_SUP,FEMMES_NON_SCOL_BAC_OU_PLUS,PART_FEMMES_SANS_DIPLOME_CEP_PCT,PART_FEMMES_BAC_OU_PLUS_PCT,PART_FEMMES_DIPLOME_SUP_PCT,PART_FEMMES_BAC5_PLUS_PCT,MANQUANT_J7A,MANQUANT_J7B,MANQUANT_J7C,NB_BLOCS_J7_MANQUANTS,POPULATION_15_64,POPULATION_15_24_ANS,POPULATION_25_54_ANS,POPULATION_55_64_ANS,POPULATION_15_64_HOMMES,POPULATION_15_64_FEMMES,ACTIFS_15_64,ACTIFS_15_24,ACTIFS_25_54,ACTIFS_55_64,ACTIFS_15_64_HOMMES,ACTIFS_15_64_FEMMES,ACTIFS_OCCUPES_15_64,ACTIFS_OCCUPES_15_24,ACTIFS_OCCUPES_25_54,ACTIFS_OCCUPES_55_64,ACTIFS_OCCUPES_15_64_HOMMES,ACTIFS_OCCUPES_15_64_FEMMES,CHOMEURS_15_64,CHOMEURS_15_24,CHOMEURS_25_54,CHOMEURS_55_64,INACTIFS_15_64,ELEVES_ETUDIANTS_STAGIAIRES_15_64,RETRAITES_PRERETRAITES_15_64,AUTRES_INACTIFS_15_64,ACTIFS_OCCUPES_15_64_COMP,ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS,ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS,ACTIFS_OCCUPES_RESIDENTS_CADRES,ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES,ACTIFS_OCCUPES_RESIDENTS_EMPLOYES,ACTIFS_OCCUPES_RESIDENTS_OUVRIERS,EMPLOIS_LT_PRINC,ACTIFS_OCCUPES_RESIDENTS_15P,EMPLOIS_15P_LT,EMPLOIS_SALARIES_LT,EMPLOIS_SALARIES_FEMMES_LT,EMPLOIS_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_NON_SALARIES_LT,EMPLOIS_NON_SALARIES_FEMMES_LT,EMPLOIS_NON_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_LT_COMP,EMPLOIS_LT_AGRICULTEURS,EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS,EMPLOIS_LT_CADRES,EMPLOIS_LT_PROF_INTERMEDIAIRES,EMPLOIS_LT_EMPLOYES,EMPLOIS_LT_OUVRIERS,EMPLOIS_LT_AGRICULTURE,EMPLOIS_LT_INDUSTRIE,EMPLOIS_LT_CONSTRUCTION,EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES,

In [50]:
#Rapporter la concurrence aux emplois

if "NB_RESTAURATION_RAPIDE" in profil_j8.columns:
    profil_j8[
        "NB_RESTAURATION_RAPIDE_1000_EMPLOIS"
    ] = taux_pour_mille(
        profil_j8[
            "NB_RESTAURATION_RAPIDE"
        ],
        profil_j8["EMPLOIS_15P_LT"],
    )

    densite_rapide_emploi_idf = (
        profil_j8[
            "NB_RESTAURATION_RAPIDE"
        ].sum()
        / profil_j8[
            "EMPLOIS_15P_LT"
        ].sum()
        * 1_000
    )

    profil_j8[
        "INDICE_DENSITE_RAPIDE_PAR_EMPLOI_IDF_BASE100"
    ] = (
        profil_j8[
            "NB_RESTAURATION_RAPIDE_1000_EMPLOIS"
        ]
        / densite_rapide_emploi_idf
        * 100
    )

    print(
        "Densité régionale de restauration rapide :",
        round(
            densite_rapide_emploi_idf,
            3,
        ),
        "pour 1 000 emplois",
    )


if "NB_RESTAURANTS_TOTAL" in profil_j8.columns:
    profil_j8[
        "NB_RESTAURANTS_TOTAL_1000_EMPLOIS"
    ] = taux_pour_mille(
        profil_j8[
            "NB_RESTAURANTS_TOTAL"
        ],
        profil_j8["EMPLOIS_15P_LT"],
    )

Densité régionale de restauration rapide : 4.034 pour 1 000 emplois


In [51]:
profil_j8

,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,POP_18_24,POP_25_39,POP_40_54,POP_55_64,POP_65_79,POP_80_PLUS,POP_FEMMES,POP_HOMMES,POP_F_0_2,POP_F_3_5,POP_F_6_10,POP_F_11_14,POP_F_15_17,POP_F_18_24,POP_F_25_39,POP_F_40_54,POP_F_55_64,POP_F_65_79,POP_F_80_PLUS,POP_H_0_2,POP_H_3_5,POP_H_6_10,POP_H_11_14,POP_H_15_17,POP_H_18_24,POP_H_25_39,POP_H_40_54,POP_H_55_64,POP_H_65_79,POP_H_80_PLUS,POP_RP2023,POP_0_17,POP_15_24,POP_15_39,POP_18_39,POP_65_PLUS,PART_FEMMES_PCT,PART_0_17_PCT,PART_15_24_PCT,PART_15_39_PCT,PART_18_39_PCT,PART_65_PLUS_PCT,NB_MENAGES,NB_MENAGES_1_PERSONNE,NB_MENAGES_HOMMES_SEULS,NB_MENAGES_FEMMES_SEULES,NB_MENAGES_AUTRES_SANS_FAMILLE,NB_MENAGES_AVEC_FAMILLES,NB_MENAGES_COUPLE_SANS_ENFANT,NB_MENAGES_COUPLE_AVEC_ENFANTS,NB_MENAGES_FAMILLE_MONOPARENTALE,POP_MENAGES,POP_15_PLUS,POP_MENAGES_15_24,POP_MENAGES_25_39,POP_MENAGES_40_54,POP_MENAGES_55_64,POP_MENAGES_65_79,POP_MENAGES_80_PLUS,POP_VIVANT_SEULE_15_24,POP_VIVANT_SEULE_25_39,POP_VIVANT_SEULE_40_54,POP_VIVANT_SEULE_55_64,POP_VIVANT_SEULE_65_79,POP_VIVANT_SEULE_80_PLUS,POP_VIVANT_EN_COUPLE_15_24,POP_VIVANT_EN_COUPLE_25_39,POP_VIVANT_EN_COUPLE_40_54,POP_VIVANT_EN_COUPLE_55_64,POP_VIVANT_EN_COUPLE_65_79,POP_VIVANT_EN_COUPLE_80_PLUS,POP_15P_MARIEE,POP_15P_PACSEE,POP_15P_CONCUBINAGE,POP_15P_VEUVE_VEUF,...,PART_HOMMES_SANS_DIPLOME_CEP_PCT,PART_HOMMES_BAC_OU_PLUS_PCT,PART_HOMMES_DIPLOME_SUP_PCT,PART_HOMMES_BAC5_PLUS_PCT,FEMMES_NON_SCOL_DIPLOME_SUP,FEMMES_NON_SCOL_BAC_OU_PLUS,PART_FEMMES_SANS_DIPLOME_CEP_PCT,PART_FEMMES_BAC_OU_PLUS_PCT,PART_FEMMES_DIPLOME_SUP_PCT,PART_FEMMES_BAC5_PLUS_PCT,MANQUANT_J7A,MANQUANT_J7B,MANQUANT_J7C,NB_BLOCS_J7_MANQUANTS,POPULATION_15_64,POPULATION_15_24_ANS,POPULATION_25_54_ANS,POPULATION_55_64_ANS,POPULATION_15_64_HOMMES,POPULATION_15_64_FEMMES,ACTIFS_15_64,ACTIFS_15_24,ACTIFS_25_54,ACTIFS_55_64,ACTIFS_15_64_HOMMES,ACTIFS_15_64_FEMMES,ACTIFS_OCCUPES_15_64,ACTIFS_OCCUPES_15_24,ACTIFS_OCCUPES_25_54,ACTIFS_OCCUPES_55_64,ACTIFS_OCCUPES_15_64_HOMMES,ACTIFS_OCCUPES_15_64_FEMMES,CHOMEURS_15_64,CHOMEURS_15_24,CHOMEURS_25_54,CHOMEURS_55_64,INACTIFS_15_64,ELEVES_ETUDIANTS_STAGIAIRES_15_64,RETRAITES_PRERETRAITES_15_64,AUTRES_INACTIFS_15_64,ACTIFS_OCCUPES_15_64_COMP,ACTIFS_OCCUPES_RESIDENTS_AGRICULTEURS,ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS,ACTIFS_OCCUPES_RESIDENTS_CADRES,ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES,ACTIFS_OCCUPES_RESIDENTS_EMPLOYES,ACTIFS_OCCUPES_RESIDENTS_OUVRIERS,EMPLOIS_LT_PRINC,ACTIFS_OCCUPES_RESIDENTS_15P,EMPLOIS_15P_LT,EMPLOIS_SALARIES_LT,EMPLOIS_SALARIES_FEMMES_LT,EMPLOIS_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_NON_SALARIES_LT,EMPLOIS_NON_SALARIES_FEMMES_LT,EMPLOIS_NON_SALARIES_TEMPS_PARTIEL_LT,EMPLOIS_LT_COMP,EMPLOIS_LT_AGRICULTEURS,EMPLOIS_LT_ARTISANS_COMMERCANTS_CHEFS,EMPLOIS_LT_CADRES,EMPLOIS_LT_PROF_INTERMEDIAIRES,EMPLOIS_LT_EMPLOYES,EMPLOIS_LT_OUVRIERS,EMPLOIS_LT_AGRICULTURE,EMPLOIS_LT_INDUSTRIE,EMPLOIS_LT_CONSTRUCTION,EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES,EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL,EMPLOIS_FEMMES_LT_COMP,CHOMEURS_15_6

In [52]:
#Afficher les principaux pôles d'emploi

colonne_nom = next(
    (
        colonne
        for colonne in [
            "LIBELLE",
            "NOM_COMMUNE",
            "LIBGEO",
        ]
        if colonne in profil_j8.columns
    ),
    None,
)

colonnes_classement = [
    "CODGEO",
]

if colonne_nom is not None:
    colonnes_classement.append(
        colonne_nom
    )

colonnes_classement += [
    "POP_RP2023",
    "EMPLOIS_15P_LT",
    "ACTIFS_OCCUPES_RESIDENTS_15P",
    "INDICE_CONCENTRATION_EMPLOI",
    "EMPLOIS_15P_LT_POUR_1000_HAB",
]

classement_poles_emploi = (
    profil_j8
    .sort_values(
        "EMPLOIS_15P_LT",
        ascending=False,
    )
    [colonnes_classement]
    .head(30)
)

display(
    classement_poles_emploi
)

,CODGEO,NOM_COMMUNE,POP_RP2023,EMPLOIS_15P_LT,ACTIFS_OCCUPES_RESIDENTS_15P,INDICE_CONCENTRATION_EMPLOI,EMPLOIS_15P_LT_POUR_1000_HAB
0,75056,Paris,2.103778e+06,1.810931e+06,1.070897e+06,169.104215,860.799646
973,92026,Courbevoie,8.290200e+04,1.142279e+05,4.298918e+04,265.713194,1377.866794
1027,93066,Saint-Denis,1.490770e+05,1.018918e+05,5.882805e+04,173.202783,683.484535
984,92050,Nanterre,9.778300e+04,9.734776e+04,4.417594e+04,220.363730,995.548891
965,92012,Boulogne-Billancourt,1.190190e+05,9.417211e+04,6.102957e+04,154.305714,791.235942
987,92062,Puteaux,4.400200e+04,9.149951e+04,2.325893e+04,393.395212,2079.439835
978,92040,Issy-les-Moulineaux,6.766900e+04,7.071559e+04,3.599281e+04,196.471431,1045.021882
1225,95527,Roissy-en-France,2.674000e+03,6.959055e+04,1.445072e+03,4815.713986,26024.889349
979,92044,Levallois-Perret,6.809200e+04,6.686121e+04,3.529940e+04,189.411724,981.924593
1016,93048,Montreuil,1.119340e+05,6.119577e+04,5.142191e+04,119.007182,546.712939


In [53]:
#Explorer emploi et concurrence

if (
    "NB_RESTAURATION_RAPIDE_1000_EMPLOIS"
    in profil_j8.columns
):
    communes_emplois_concurrence = (
        profil_j8[
            profil_j8[
                "EMPLOIS_15P_LT"
            ]
            >= 5_000
        ]
        .sort_values(
            "NB_RESTAURATION_RAPIDE_1000_EMPLOIS",
            ascending=True,
        )
        [
            [
                colonne
                for colonne in [
                    "CODGEO",
                    colonne_nom,
                    "POP_RP2023",
                    "EMPLOIS_15P_LT",
                    "INDICE_CONCENTRATION_EMPLOI",
                    "NB_RESTAURATION_RAPIDE",
                    "NB_RESTAURATION_RAPIDE_1000_EMPLOIS",
                    "PART_EMPLOIS_TERTIAIRES_LT_PCT",
                ]
                if colonne is not None
                and colonne in profil_j8.columns
            ]
        ]
        .head(30)
    )

    display(
        communes_emplois_concurrence
    )

,CODGEO,NOM_COMMUNE,POP_RP2023,EMPLOIS_15P_LT,INDICE_CONCENTRATION_EMPLOI,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_RAPIDE_1000_EMPLOIS,PART_EMPLOIS_TERTIAIRES_LT_PCT
1225,95527,Roissy-en-France,2674.00002,69590.55464,4815.713986,19,0.273026,95.478908
910,91534,Saclay,4437.00000,6562.72288,315.515523,2,0.304752,75.924702
279,77291,Le Mesnil-Amelot,991.99997,6064.09257,1189.245061,3,0.494715,92.550100
1068,94065,Rungis,5611.00000,26300.36154,987.839021,17,0.646379,88.496925
958,91689,Wissous,7150.99998,8952.85051,257.712237,6,0.670178,79.733025
986,92060,Le Plessis-Robinson,28848.00000,15655.95647,108.004983,11,0.702608,80.249150
105,77111,Chessy,7770.99996,16477.71225,395.907336,12,0.728256,95.695087
140,77146,Croissy-Beaubourg,2017.99999,6677.39763,777.005845,5,0.748795,71.146256
548,78117,Buc,5816.00001,5898.16785,214.230782,6,1.017265,66.865013
975,92033,Garches,17742.99994,5395.54366,66.520100,6,1.112029,92.227417


In [54]:
#Contrôles finaux

assert len(profil_j8) == len(profil_j7)
assert profil_j8["CODGEO"].is_unique
assert profil_j8["CODGEO"].notna().all()

colonnes_obligatoires_j8 = [
    "POPULATION_15_64",
    "ACTIFS_15_64",
    "ACTIFS_OCCUPES_15_64",
    "CHOMEURS_15_64",
    "EMPLOIS_15P_LT",
    "ACTIFS_OCCUPES_RESIDENTS_15P",
    "INDICE_CONCENTRATION_EMPLOI",
]

for colonne in colonnes_obligatoires_j8:
    assert profil_j8[colonne].notna().all()

colonnes_taux_j8 = [
    colonne
    for colonne in indicateurs_j8.columns
    if (
        colonne.startswith("TAUX_")
        or colonne.startswith("PART_")
    )
]

for colonne in colonnes_taux_j8:
    valeurs = indicateurs_j8[
        colonne
    ].dropna()

    assert valeurs.between(
        -0.01,
        100.01,
    ).all(), (
        f"Valeurs hors intervalle dans {colonne}"
    )

assert "75056" in profil_j8["CODGEO"].values
assert "93066" in profil_j8["CODGEO"].values
assert "93059" not in profil_j8["CODGEO"].values

print("Tous les contrôles J8 sont validés ✅")

Tous les contrôles J8 sont validés ✅


In [55]:
#Enregistrer les résultats finaux

enregistrer_csv(
    profil_j8,
    FICHIER_PROFIL_J8,
)

FICHIER_CLASSEMENT_EMPLOIS = (
    DOSSIER_INTERIM
    / "j8_classement_poles_emploi.csv"
)

enregistrer_csv(
    classement_poles_emploi,
    FICHIER_CLASSEMENT_EMPLOIS,
)

if (
    "communes_emplois_concurrence"
    in globals()
):
    FICHIER_EMPLOIS_CONCURRENCE = (
        DOSSIER_INTERIM
        / "j8_communes_emplois_concurrence.csv"
    )

    enregistrer_csv(
        communes_emplois_concurrence,
        FICHIER_EMPLOIS_CONCURRENCE,
    )

print("Profil J8 :", FICHIER_PROFIL_J8)
print("J8 terminé ✅")

Fichier créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j8.csv
Fichier créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j8_classement_poles_emploi.csv
Fichier créé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\interim\j8_communes_emplois_concurrence.csv
Profil J8 : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j8.csv
J8 terminé ✅
